# Lab 3: Price and dividend

December yields, after the daily return

The daily lab asked whether yesterday’s SPY return forecasts today’s.
This hour uses a different series. The S&P 500 price, scaled by the
trailing twelve-month dividend, is a December object (Cochrane 2011).
There is no Blackboard submission. Bring the two (R^2) values from Task
2, and one sentence on what the regression does not claim.

The file is `price_dividend_ratio.parquet`, loaded with
`load_price_dividend()`. Yields are in percent. A Treasury `PX_LAST` in
this file is a yield. Do not call `pct_change()` on it.

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab03_price_dividend.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

for _root in (Path("scripts"), Path("../scripts"), Path("resources"), Path("../resources")):
    _p = _root.resolve()
    if _p.is_dir() and (_p / "bloomberg_loader.py").exists():
        sys.path.insert(0, str(_p))
        break

try:
    from bloomberg_loader import load_price_dividend
except ImportError:
    def load_price_dividend(tickers=None, fields=None, frequency=None):
        url = (
            "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
            "main/resources/price_dividend_ratio.parquet"
        )
        frame = pd.read_parquet(url)
        if tickers is not None:
            upper = {t.strip().upper() for t in tickers}
            frame = frame[frame["ticker"].astype(str).str.upper().isin(upper)]
        if fields is not None:
            frame = frame[frame["field"].isin(fields)]
        if frequency is not None:
            frame = frame[frame["frequency"].eq(frequency)]
        return frame.reset_index(drop=True)

## Task 1: The price and the yield

Month-end S&P 500. The yield starts in December 1970. The price starts
earlier. Plot both from the first date they share.

In [ ]:
raw = load_price_dividend(tickers=["SPX"], frequency="monthly")

def field(frame, ticker, name):
    s = frame[(frame["ticker"] == ticker) & (frame["field"] == name)].set_index("date")["value"]
    s = s.sort_index()
    return s[~s.index.duplicated()]

price = field(raw, "SPX", "PX_LAST")
div_yield = field(raw, "SPX", "EQY_DVD_YLD_12M")
both = pd.concat({"price": price, "yield": div_yield}, axis=1).dropna()
print(f"{both.index.min().date()} to {both.index.max().date()}, n={len(both)}")
print(f"Yield range: {both['yield'].min():.2f}% to {both['yield'].max():.2f}%")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(both.index, both["price"], linewidth=0.8, color="steelblue")
axes[0].set_ylabel("Price index")
axes[1].plot(both.index, both["yield"], linewidth=0.8, color="coral")
axes[1].set_ylabel("Dividend yield (%)")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

The upper panel is the wandering price from the daily lab, at a monthly
frequency. The lower panel is that price scaled by the dividend, written
as a yield.

## Task 2: Two samples, one regression

December rows only. (D/P) is the yield divided by 100. The total return
is the change in `TOT_RETURN_INDEX_GROSS_DVDS`. Fit next year’s total
return on this December’s (D/P), twice: predictors through 2009, then
through 2024.

In [ ]:
pd_ = load_price_dividend(frequency="monthly")
dec = pd.DataFrame({
    "dy": field(pd_, "SPX", "EQY_DVD_YLD_12M"),
    "tr": field(pd_, "SPX", "TOT_RETURN_INDEX_GROSS_DVDS"),
    "pe": field(pd_, "SPX", "PE_RATIO"),
    "bill": field(pd_, "USGG3M", "PX_LAST"),
    "P": field(pd_, "SPX", "PX_LAST"),
})
dec = dec[dec.index.month == 12].copy()
dec["dp"] = dec["dy"] / 100
dec["ep"] = 1 / dec["pe"]
dec["D"] = dec["dp"] * dec["P"]
dec["R"] = dec["tr"].pct_change()
dec["ex"] = dec["R"] - dec["bill"] / 100
dec["dg"] = dec["D"].pct_change()
dec["R_next"] = dec["R"].shift(-1)
dec["ex_next"] = dec["ex"].shift(-1)
dec["dg_next"] = dec["dg"].shift(-1)

def fit(frame, y, x):
    sample = frame[[y, x]].dropna()
    model = sm.OLS(sample[y], sm.add_constant(sample[x])).fit()
    print(
        f"{y} on {x}: n={int(model.nobs)} "
        f"{sample.index.min().year}-{sample.index.max().year} "
        f"b={model.params[x]:.2f} t={model.tvalues[x]:.2f} "
        f"R2={model.rsquared:.3f}"
    )
    return model

print("Through 2009")
fit(dec.loc[: "2009-12-31"], "R_next", "dp")
print("Through 2024")
fit(dec.loc[: "2024-12-31"], "R_next", "dp")
fit(dec.loc[: "2024-12-31"], "dg_next", "dp")
fit(dec.loc[: "2024-12-31"], "ex_next", "ep")
same = dec.loc[: "2024-12-31"].dropna(subset=["dp"])
fit(same, "ex_next", "ex")

Cochrane’s CRSP excess-return regression, 1947 to 2009, has (R^2 =
0.09). Your first (R^2) is a total return on the S&P 500, not that
regression. The second (R^2) is the same specification carried through
2024. Write one sentence on which fit you would quote, and why the
dividend-growth (R^2) matters for the identity.

## Task 3: Three portfolios

Utilities, information technology, and financials, from 1990. The dashed
line is December 2008.

In [ ]:
sectors = load_price_dividend(frequency="monthly")
fig, ax = plt.subplots(figsize=(10, 4.5))
labels = {
    "SPX": "S&P 500",
    "S5UTIL": "Utilities",
    "S5INFT": "Information technology",
    "S5FINL": "Financials",
}
for ticker, label in labels.items():
    s = field(sectors, ticker, "EQY_DVD_YLD_12M").loc["1990-01-01":]
    ax.plot(s.index, s, linewidth=0.9, label=label)
ax.axvline(pd.Timestamp("2008-12-31"), color="gray", linestyle="--", linewidth=0.6)
ax.set_ylabel("Dividend yield (%)")
ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

fin = field(sectors, "S5FINL", "EQY_DVD_YLD_12M")
print("Financials, December 2008 and December 2009:")
print(fin.loc[["2008-12-31", "2009-12-31"]].round(2))

Technology’s yield is the reason the price–dividend ratio is a poor
scale for that sector. Financials are the case where the dividend itself
moved.

Cochrane, John H. 2011. “Presidential Address: Discount Rates.” *Journal
of Finance* 66 (4): 1047–1108.
<https://doi.org/10.1111/j.1540-6261.2011.01671.x>.